# 🔬 Module 1.2: Pixels and Channels — The Atomic Units of Images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_01_Image_Foundations/02_Pixels_And_Channels/pixels_and_channels.ipynb)

---

## 🎯 Learning Objectives
1. Deep understanding of pixel structure and memory layout
2. Channel separation, combination, and manipulation
3. Bit-depth and its effect on image quality
4. Pixel neighborhoods and their role in RL state representation

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless pillow -q

import numpy as np
import matplotlib.pyplot as plt
import cv2
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision
import torchvision.transforms as transforms

# Real images from scikit-image (built-in, no download needed)
astronaut_img = data.astronaut()       # 512x512x3 real photo
camera_img = data.camera()             # 512x512 grayscale real photo  
coins_img = data.coins()               # Real coins photograph
coffee_img = data.coffee()             # 400x600x3 real photo

# MNIST - 70,000 real handwritten digits (auto-downloads ~11MB)
mnist_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True)
print(f"✅ MNIST loaded: {len(mnist_dataset)} real handwritten digit images (28x28)")
print(f"✅ scikit-image loaded: astronaut {astronaut_img.shape}, camera {camera_img.shape}")

## Deep Mathematical Derivation: Pixel Arithmetic and Color Theory

### Step 1: Color Mixing (Additive Model)
For RGB display, perceived color $C$ is:
$$C = r \cdot \mathbf{R} + g \cdot \mathbf{G} + b \cdot \mathbf{B}$$

where $\mathbf{R}, \mathbf{G}, \mathbf{B}$ are basis vectors in CIE XYZ space.

### Step 2: Gamma Correction (Non-linear Encoding)
Human vision is logarithmic, so displays use gamma encoding:
$$V_{\text{encoded}} = V_{\text{linear}}^{1/\gamma}, \quad \gamma \approx 2.2$$

**Full sRGB transfer function:**
$$C_{\text{sRGB}} = \begin{cases} 12.92 \cdot C_{\text{linear}} & C_{\text{linear}} \leq 0.0031308 \\ 1.055 \cdot C_{\text{linear}}^{1/2.4} - 0.055 & C_{\text{linear}} > 0.0031308 \end{cases}$$

### Step 3: Pixel Arithmetic as Linear Algebra
**Brightness:** $\mathbf{p}' = \mathbf{p} + \beta \cdot \mathbf{1}$ (translation in color space)

**Contrast:** $\mathbf{p}' = \alpha(\mathbf{p} - \boldsymbol{\mu}) + \boldsymbol{\mu}$ (scaling about mean)

**Color matrix:** $\mathbf{p}' = \mathbf{M} \cdot \mathbf{p}$ where $\mathbf{M} \in \mathbb{R}^{3\times3}$

**Proof that saturation is a matrix operation:**
$$\mathbf{M}_{\text{sat}}(s) = \begin{bmatrix} 0.213 + 0.787s & 0.715 - 0.715s & 0.072 - 0.072s \\ 0.213 - 0.213s & 0.715 + 0.285s & 0.072 - 0.072s \\ 0.213 - 0.213s & 0.715 - 0.715s & 0.072 + 0.928s \end{bmatrix}$$

At $s=0$: grayscale (luminance only). At $s=1$: identity. Derived from ITU-R BT.709 luminance weights.

### Step 4: Information Content of a Pixel
**Shannon entropy** of a pixel distribution:
$$H = -\sum_{k=0}^{255} p(k) \log_2 p(k) \leq \log_2(256) = 8 \text{ bits}$$

**Equality holds iff** $p(k) = 1/256$ for all $k$ (uniform distribution).

### RL Connection: Action Space Size
If each channel has $n$ possible adjustments (e.g., +10, -10, +20, -20, 0):
$$|\mathcal{A}| = n_R \times n_G \times n_B$$

With 5 adjustments per channel: $|\mathcal{A}| = 5^3 = 125$ possible actions per step.

## Color Theory — Additive vs. Subtractive Mixing from Physics

### Step 1: The Physics of Light Emission (Additive Mixing)

When two light sources with spectral power distributions $S_1(\lambda)$ and $S_2(\lambda)$ illuminate the same point, the resulting spectrum is their sum:

$$S_{\text{combined}}(\lambda) = S_1(\lambda) + S_2(\lambda)$$

**Proof from linearity of electromagnetic superposition:** Maxwell's equations are linear, so electric fields add: $\mathbf{E}_{\text{total}} = \mathbf{E}_1 + \mathbf{E}_2$. Since intensity $I \propto |\mathbf{E}|^2$ and the sources are incoherent (random phases), the time-averaged intensity is:

$$\langle I \rangle = \langle |\mathbf{E}_1 + \mathbf{E}_2|^2 \rangle = \langle |\mathbf{E}_1|^2 \rangle + \langle |\mathbf{E}_2|^2 \rangle = I_1 + I_2 \quad \blacksquare$$

This is why **RGB displays use additive mixing**: each pixel emits three wavelengths that add together.

### Step 2: Subtractive Mixing (Pigments and Filters)

A pigment with transmittance $T(\lambda)$ selectively absorbs wavelengths:
$$S_{\text{transmitted}}(\lambda) = S_{\text{incident}}(\lambda) \cdot T(\lambda)$$

Two pigments layered together:
$$S_{\text{out}}(\lambda) = S_{\text{in}}(\lambda) \cdot T_1(\lambda) \cdot T_2(\lambda)$$

This is **multiplicative** in the spectral domain — hence "subtractive" (each filter removes more light).

**CMY(K) model:** Cyan absorbs Red, Magenta absorbs Green, Yellow absorbs Blue:
$$\begin{bmatrix} C \\ M \\ Y \end{bmatrix} = \begin{bmatrix} 1 \\ 1 \\ 1 \end{bmatrix} - \begin{bmatrix} R \\ G \\ B \end{bmatrix}$$

### Step 3: Why Additive Primary Colors Are Red, Green, Blue

The choice of RGB primaries is NOT arbitrary — it derives from the spectral sensitivity peaks of human cone photoreceptors:

$$L(\lambda) \text{ peaks at } \sim 564\text{nm (red-sensitive)}$$
$$M(\lambda) \text{ peaks at } \sim 534\text{nm (green-sensitive)}$$
$$S(\lambda) \text{ peaks at } \sim 420\text{nm (blue-sensitive)}$$

By Grassmann's laws of additive color mixture:
1. **Additivity:** If $A \sim A'$ and $B \sim B'$, then $A + B \sim A' + B'$ (where $\sim$ means "matches in color")
2. **Scalar multiplication:** If $A \sim A'$, then $\alpha A \sim \alpha A'$ for $\alpha > 0$
3. **Transitivity:** If $A \sim B$ and $B \sim C$, then $A \sim C$

These laws establish that color matching is a **linear** operation in 3D, which is why a $3 \times 3$ matrix can transform between color spaces.

### RL Interpretation

For an RL agent manipulating colors, the choice between additive and multiplicative operations fundamentally changes the action space geometry:
- Additive actions: translations in color space (brightness adjustment)
- Multiplicative actions: scaling in color space (contrast/color balance)
- The agent must learn which type of operation is appropriate for each task

## Gamma Correction — Derivation from the Weber-Fechner Law

Gamma correction is not an arbitrary mapping — it is a direct consequence of how human perception works, derived from 19th-century psychophysics.

### Step 1: Weber-Fechner Law of Perception

**Weber's Law (1834):** The just-noticeable difference (JND) in a stimulus is proportional to the stimulus magnitude:
$$\frac{\Delta I}{I} = k \quad (\text{constant})$$

where $I$ is the stimulus intensity and $\Delta I$ is the smallest detectable change.

**Fechner's Integration (1860):** Integrating Weber's law gives the perceived intensity $P$ as a function of physical intensity $I$:

$$dP = k \cdot \frac{dI}{I} \implies P = k \ln I + C$$

**The human visual system perceives intensity logarithmically, not linearly.**

### Step 2: Stevens' Power Law (Modern Refinement)

Stevens (1957) showed that a power law fits perception data better:
$$P = k \cdot I^\gamma, \quad \gamma \approx 0.3-0.5 \text{ for brightness}$$

For the specific case of brightness perception: $\gamma \approx 1/3$ (cube root law), which is why the CIELAB L* channel uses:
$$L^* = 116 \cdot (Y/Y_n)^{1/3} - 16$$

### Step 3: Gamma Encoding Derivation

**Goal:** Given limited bit depth ($L = 256$ levels), allocate quantization levels to minimize perceptual error.

**Uniform quantization in linear space:** step size $\Delta I = I_{\max}/L$

The perceptual step varies across the range:
- At $I = I_{\max}$: $\Delta P \propto \Delta I / I_{\max} = 1/L$ (barely visible)
- At $I = I_{\max}/100$: $\Delta P \propto \Delta I / (I_{\max}/100) = 100/L$ (100× more visible)

**Solution:** Encode the signal with a power law to equalize perceptual steps:

$$V_{\text{encoded}} = V_{\text{linear}}^{1/\gamma}$$

**Proof of perceptual uniformity:** The quantization step in encoded space is $\Delta V = 1/L$ (uniform). In linear space:

$$\Delta I = \frac{dI}{dV}\Delta V = \gamma V^{\gamma-1} \Delta V$$

The perceptual step: $\Delta P \propto \Delta I / I = \gamma V^{\gamma-1} / V^\gamma \cdot \Delta V = \gamma/V \cdot \Delta V$

Wait — this isn't constant. The exact constant-perception encoding would be $V = \ln I$. Gamma encoding is a practical approximation that is simpler to decode. $\blacksquare$

### Step 4: The sRGB Transfer Function

The full sRGB standard (IEC 61966-2-1) uses a piecewise function that approximates $\gamma = 2.2$:

$$C_{\text{sRGB}} = \begin{cases} 12.92 \cdot C_{\text{linear}} & C_{\text{linear}} \leq 0.0031308 \\ 1.055 \cdot C_{\text{linear}}^{1/2.4} - 0.055 & C_{\text{linear}} > 0.0031308 \end{cases}$$

The linear segment near zero avoids the infinite slope of $x^{1/2.4}$ at $x = 0$, which would waste quantization levels on imperceptible near-black differences.

### Step 5: Why RL Agents Must Be Gamma-Aware

An RL agent that naively adds $+10$ to pixel values creates a LARGER perceptual change in dark regions than bright regions (because of gamma encoding). For perceptually uniform actions, the agent should operate in linear space or adjust its step size based on current intensity.

## 1. Pixel: The Fundamental Unit

### Definition:
A **pixel** (picture element) is the smallest addressable element in an image.

### Grayscale Pixel:
$$p \in \{0, 1, 2, ..., 2^k - 1\}$$

For 8-bit: $p \in \{0, 1, ..., 255\}$ where:
- $p = 0$ → Black (no light)
- $p = 255$ → White (maximum light)

### Color Pixel (RGB):
$$\mathbf{p} = (r, g, b) \in \{0,...,255\}^3$$

Total possible colors: $256^3 = 16,777,216$ (16.7 million colors)

### Memory Layout:
$$\text{Memory} = H \times W \times C \times \text{bytes\_per\_channel}$$

## Memory Layout and Data Types — Mathematical Implications

The way pixels are stored in memory has direct mathematical consequences for image operations and RL state representation.

### Step 1: Row-Major vs. Column-Major Storage

**Row-major (C/NumPy):** $I[i, j, c]$ is stored at offset $i \times W \times C + j \times C + c$

**Column-major (Fortran/MATLAB):** $I[i, j, c]$ is stored at offset $c \times H \times W + j \times H + i$

**Channel-first (PyTorch):** $I[c, i, j]$ — optimized for convolution operations

$$I_{\text{CHW}}[c, i, j] = I_{\text{HWC}}[i, j, c]$$

### Step 2: Integer vs. Floating-Point Representation

| Data Type | Range | Memory | Precision |
|:----------|:------|:-------|:----------|
| uint8 | $[0, 255]$ | 1 byte | 256 levels |
| uint16 | $[0, 65535]$ | 2 bytes | 65536 levels |
| float32 | $[\sim 10^{-38}, \sim 10^{38}]$ | 4 bytes | ~7 decimal digits |
| float64 | $[\sim 10^{-308}, \sim 10^{308}]$ | 8 bytes | ~15 decimal digits |

### Step 3: Overflow and Saturation in Pixel Arithmetic

**Problem:** Adding two uint8 pixels: $200 + 100 = 300 > 255$

**Modular arithmetic (wrap-around):** $200 + 100 \equiv 44 \pmod{256}$ — creates artifacts!

**Saturating arithmetic:** $\min(200 + 100, 255) = 255$ — preserves brightness ordering.

**Proof that saturating arithmetic preserves ordering:**
If $a \leq b$, then $\text{sat}(a + c) = \min(a+c, 255) \leq \min(b+c, 255) = \text{sat}(b+c)$ since $a + c \leq b + c$. $\blacksquare$

OpenCV uses saturating arithmetic by default. NumPy uses modular arithmetic — a common source of bugs in image processing code.

### Step 4: Normalization Conventions

Different frameworks use different value ranges:

$$I_{\text{uint8}} \in [0, 255] \xleftrightarrow{\div 255} I_{\text{float}} \in [0.0, 1.0] \xleftrightarrow{\times 2 - 1} I_{\text{norm}} \in [-1.0, 1.0]$$

**For RL agents:** The normalization affects gradient magnitudes in neural networks. Standard practice:
$$I_{\text{RL}} = \frac{I_{\text{uint8}}}{255.0} \quad \text{or} \quad I_{\text{RL}} = \frac{I_{\text{uint8}} - \mu}{\sigma}$$

The latter (z-score normalization with dataset statistics) gives zero mean and unit variance, which helps neural network training converge faster.

In [ ]:
# Create a tiny 8x8 image to study pixel structure
tiny_img = np.zeros((8, 8, 3), dtype=np.uint8)

# Create a pattern with known values
# Row 0: Pure Red gradient
tiny_img[0, :, 0] = np.linspace(0, 255, 8).astype(np.uint8)
# Row 1: Pure Green gradient  
tiny_img[1, :, 1] = np.linspace(0, 255, 8).astype(np.uint8)
# Row 2: Pure Blue gradient
tiny_img[2, :, 2] = np.linspace(0, 255, 8).astype(np.uint8)
# Row 3: Yellow (R+G)
tiny_img[3, :, 0] = 255
tiny_img[3, :, 1] = np.linspace(0, 255, 8).astype(np.uint8)
# Row 4: Cyan (G+B)
tiny_img[4, :, 1] = 255
tiny_img[4, :, 2] = np.linspace(0, 255, 8).astype(np.uint8)
# Row 5: Magenta (R+B)
tiny_img[5, :, 0] = 255
tiny_img[5, :, 2] = np.linspace(0, 255, 8).astype(np.uint8)
# Row 6: White gradient
for c in range(3):
    tiny_img[6, :, c] = np.linspace(0, 255, 8).astype(np.uint8)
# Row 7: Random
tiny_img[7] = np.random.randint(0, 256, (8, 3), dtype=np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Full color
axes[0].imshow(tiny_img, interpolation='nearest')
axes[0].set_title('8×8 Color Image\n(24 bits per pixel)')
axes[0].set_xticks(range(8))
axes[0].set_yticks(range(8))
axes[0].grid(True, alpha=0.3)

# Individual channels
channel_names = ['Red', 'Green', 'Blue']
cmaps = ['Reds', 'Greens', 'Blues']

for ch in range(3):
    axes[ch+1].imshow(tiny_img[:,:,ch], cmap=cmaps[ch], vmin=0, vmax=255, interpolation='nearest')
    axes[ch+1].set_title(f'{channel_names[ch]} Channel\n(8 bits per pixel)')
    axes[ch+1].set_xticks(range(8))
    axes[ch+1].set_yticks(range(8))
    axes[ch+1].grid(True, alpha=0.3)
    
    # Show values
    for i in range(8):
        for j in range(8):
            val = tiny_img[i, j, ch]
            color = 'white' if val < 128 else 'black'
            axes[ch+1].text(j, i, str(val), ha='center', va='center',
                          fontsize=7, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('pixel_channels_detail.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Single pixel at (3, 5): RGB = {tuple(tiny_img[3, 5])}")
print(f"   This means: Red={tiny_img[3,5,0]}, Green={tiny_img[3,5,1]}, Blue={tiny_img[3,5,2]}")
print(f"   Memory for this pixel: 3 bytes = 24 bits")
print(f"\n📊 Full image memory: {tiny_img.nbytes} bytes = {8}×{8}×{3} = {8*8*3} bytes")

## Pixel Distance Metrics — Mathematical Foundations for RL State Comparison

Measuring the "distance" between two pixel values (or between two images) is fundamental to RL reward design. Different metrics capture different aspects of similarity.

### Step 1: Euclidean Distance in RGB Space

$$d_E(\mathbf{p}_1, \mathbf{p}_2) = \sqrt{(R_1 - R_2)^2 + (G_1 - G_2)^2 + (B_1 - B_2)^2}$$

**Problem:** RGB Euclidean distance does NOT correlate well with perceptual difference. A $\Delta R = 50$ is more noticeable than a $\Delta B = 50$ because the eye is more sensitive to green/red.

### Step 2: Weighted Euclidean (Low-Cost Perceptual Approximation)

Approximate perceptual weighting:
$$d_W(\mathbf{p}_1, \mathbf{p}_2) = \sqrt{w_R \Delta R^2 + w_G \Delta G^2 + w_B \Delta B^2}$$

The Compuphase approximation (fast, reasonably accurate):
$$\bar{R} = \frac{R_1 + R_2}{2}$$
$$d \approx \sqrt{(2 + \bar{R}/256)\Delta R^2 + 4\Delta G^2 + (2 + (255-\bar{R})/256)\Delta B^2}$$

**Derivation:** The weights are derived from the varying sensitivity of the human eye to color differences as a function of the mean red value — based on empirical psychophysical measurements.

### Step 3: Manhattan ($L^1$) and Chebyshev ($L^\infty$) Distances

**Manhattan distance:**
$$d_1(\mathbf{p}_1, \mathbf{p}_2) = |R_1 - R_2| + |G_1 - G_2| + |B_1 - B_2|$$

**Chebyshev distance:**
$$d_\infty(\mathbf{p}_1, \mathbf{p}_2) = \max(|R_1 - R_2|, |G_1 - G_2|, |B_1 - B_2|)$$

**General $L^p$ norm:**
$$d_p(\mathbf{p}_1, \mathbf{p}_2) = \left(\sum_{c \in \{R,G,B\}} |c_1 - c_2|^p\right)^{1/p}$$

**Relationship:** $d_\infty \leq d_2 \leq d_1 \leq 3 \cdot d_\infty$

**Proof:** For $\mathbf{x} \in \mathbb{R}^n$: $\|\mathbf{x}\|_\infty \leq \|\mathbf{x}\|_2 \leq \|\mathbf{x}\|_1 \leq \sqrt{n}\|\mathbf{x}\|_2 \leq n\|\mathbf{x}\|_\infty$. With $n = 3$: $d_1 \leq \sqrt{3} \cdot d_2 \leq 3 \cdot d_\infty$. $\blacksquare$

### Step 4: Earth Mover's Distance (EMD) for Image Comparison

For comparing entire images via their color histograms, EMD measures the minimum "work" to transform one histogram into another:

$$\text{EMD}(h_1, h_2) = \min_{\{f_{ij}\}} \frac{\sum_{i,j} f_{ij} \cdot d(i, j)}{\sum_{i,j} f_{ij}}$$

where $f_{ij}$ is the flow from bin $i$ of $h_1$ to bin $j$ of $h_2$, subject to flow conservation constraints.

### Step 5: RL Reward Design Implications

The choice of distance metric directly affects RL agent behavior:
- **$L^2$ (MSE):** Agent minimizes pixel-level error → may produce blurry results
- **$L^1$ (MAE):** Agent produces sharper images but with more artifacts
- **Perceptual (LPIPS):** Agent optimizes for human perception → best subjective quality
- **$\Delta E$ (CIELAB):** Agent optimizes for calibrated color accuracy

The optimal reward often combines multiple metrics: $R = -\alpha \cdot \text{MSE} - \beta \cdot (1 - \text{SSIM}) - \gamma \cdot \text{LPIPS}$

## 2. Pixel Neighborhoods — Critical for RL State

### 4-Connected Neighborhood:
$$N_4(p) = \{(x-1,y), (x+1,y), (x,y-1), (x,y+1)\}$$

### 8-Connected Neighborhood:
$$N_8(p) = N_4(p) \cup \{(x-1,y-1), (x-1,y+1), (x+1,y-1), (x+1,y+1)\}$$

### Why Neighborhoods Matter for RL:
When an RL agent makes decisions about a pixel, it needs **context** from neighboring pixels. The neighborhood defines the agent's **local observation**.

In [ ]:
# Visualize pixel neighborhoods
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Create a sample grayscale patch
patch = np.random.randint(50, 200, (7, 7), dtype=np.uint8)
center = (3, 3)

for ax_idx, (title, neighbors) in enumerate([
    ('Center Pixel', [(3,3)]),
    ('4-Connected N₄(p)', [(2,3),(4,3),(3,2),(3,4)]),
    ('8-Connected N₈(p)', [(2,3),(4,3),(3,2),(3,4),(2,2),(2,4),(4,2),(4,4)])
]):
    ax = axes[ax_idx]
    ax.imshow(patch, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    
    # Highlight neighbors
    for (i, j) in neighbors:
        rect = Rectangle((j-0.5, i-0.5), 1, 1, fill=False, 
                         edgecolor='red', linewidth=2)
        ax.add_patch(rect)
    
    # Always highlight center
    rect = Rectangle((2.5, 2.5), 1, 1, fill=False, 
                     edgecolor='yellow', linewidth=3)
    ax.add_patch(rect)
    
    # Show values
    for i in range(7):
        for j in range(7):
            color = 'white' if patch[i,j] < 128 else 'black'
            ax.text(j, i, str(patch[i,j]), ha='center', va='center',
                   fontsize=9, color=color)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(range(7))
    ax.set_yticks(range(7))
    ax.grid(True, alpha=0.3)

plt.suptitle('Pixel Neighborhoods → RL Agent\'s Local State', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('pixel_neighborhoods.png', dpi=150, bbox_inches='tight')
plt.show()

## Convex Combination and Image Blending — Algebraic Properties

Image blending is one of the most fundamental pixel-level operations. Understanding its mathematical properties enables RL agents to plan multi-step blending strategies.

### Step 1: Alpha Blending as Convex Combination

Blending two images with weight $\alpha \in [0, 1]$:
$$I_{\text{blend}}(x,y) = \alpha \cdot I_1(x,y) + (1-\alpha) \cdot I_2(x,y)$$

**Proof that the result is a valid image:**
If $I_1, I_2 \in [0, 1]^{H \times W \times C}$ and $\alpha \in [0, 1]$:
$$0 \leq \alpha \cdot I_1 + (1-\alpha) \cdot I_2 \leq \alpha \cdot 1 + (1-\alpha) \cdot 1 = 1 \quad \blacksquare$$

The blended image stays within the valid intensity range — no clipping needed.

### Step 2: Multi-Image Blending (Exposure Fusion)

For $K$ images with weights $w_1, \ldots, w_K$ (summing to 1):
$$I_{\text{fused}} = \sum_{k=1}^{K} w_k \cdot I_k$$

**Proof of associativity:** Blending $I_1$ and $I_2$ with $\alpha$, then blending the result with $I_3$ with $\beta$:

$$I_{12} = \alpha I_1 + (1-\alpha) I_2$$
$$I_{123} = \beta I_{12} + (1-\beta) I_3 = \alpha\beta I_1 + (1-\alpha)\beta I_2 + (1-\beta) I_3$$

This is a convex combination with weights $(\alpha\beta, (1-\alpha)\beta, 1-\beta)$ summing to 1. $\blacksquare$

### Step 3: Laplacian Pyramid Blending

For seamless blending with spatial variation, use multi-scale decomposition:

1. Build Laplacian pyramids $L_1, L_2$ of both images
2. Build Gaussian pyramid $G_M$ of the mask
3. Blend at each level: $L_{\text{blend}}^k = G_M^k \cdot L_1^k + (1 - G_M^k) \cdot L_2^k$
4. Reconstruct from the blended pyramid

**Why this works:** The mask is smoothed at coarser levels, so the blending boundary is smooth in low frequencies (no visible seam) while sharp in high frequencies (preserves detail).

### Step 4: Mathematical Properties of Pixel Operations

| Operation | Formula | Range Preservation | Associative | Commutative |
|:----------|:--------|:------------------|:-----------|:------------|
| Addition | $I_1 + I_2$ | No (needs clipping) | Yes | Yes |
| Multiplication | $I_1 \cdot I_2$ | Yes (if $\in [0,1]$) | Yes | Yes |
| Blending | $\alpha I_1 + (1-\alpha)I_2$ | Yes | No* | No |
| Max | $\max(I_1, I_2)$ | Yes | Yes | Yes |
| Min | $\min(I_1, I_2)$ | Yes | Yes | Yes |

*Blending is not associative in general (different $\alpha$ values).

### RL Agent Action Design

These algebraic properties constrain how the RL agent can compose pixel operations:
- **Commutative operations** (multiply, max, min): action order doesn't matter → simpler planning
- **Non-commutative** (blend with different $\alpha$): order matters → requires sequential reasoning
- **Range-preserving** operations avoid the need for clipping, producing more predictable state transitions

## Image Interpolation and Resampling — Pixel-Level Mathematics

When an RL agent resizes, rotates, or warps an image, it must compute pixel values at non-integer positions. This section covers the mathematical theory of pixel interpolation.

### Step 1: The Resampling Problem

Given an image $I[m, n]$ defined at integer coordinates, estimate $I(x, y)$ at arbitrary real coordinates $(x, y)$.

**Ideal reconstruction** (Whittaker-Shannon):
$$I(x, y) = \sum_{m} \sum_{n} I[m, n] \cdot \text{sinc}(x - m) \cdot \text{sinc}(y - n)$$

This requires an infinite sum — impractical. Practical methods use local kernels.

### Step 2: Nearest-Neighbor Interpolation

$$I_{\text{NN}}(x, y) = I[\text{round}(x), \text{round}(y)]$$

**Properties:**
- $O(1)$ computation per pixel
- Preserves original pixel values exactly (no new values created)
- Produces blocky artifacts (discontinuous)
- Error: $O(\Delta x)$ — first-order accurate

**Proof of pixel-value preservation:** Since round maps integers to themselves, $I_{\text{NN}}(m, n) = I[m, n]$ for all integer coordinates. $\blacksquare$

### Step 3: Bilinear Interpolation (Recap with Properties)

$$I_{\text{BL}}(x, y) = (1-\alpha)(1-\beta)I[m,n] + \alpha(1-\beta)I[m+1,n] + (1-\alpha)\beta I[m,n+1] + \alpha\beta I[m+1,n+1]$$

**Properties:**
- Continuous but not smooth ($C^0$, not $C^1$) — gradient has jumps at integer boundaries
- Introduces new values not in the original image (low-pass filtering effect)
- Error: $O(\Delta x^2)$ — second-order accurate

**Proof of continuity:** Each of the four weight functions $(1-\alpha)(1-\beta)$, etc. is continuous in $(\alpha, \beta) = (x - \lfloor x \rfloor, y - \lfloor y \rfloor)$. The interpolated value is a continuous function of continuous weights → continuous. $\blacksquare$

### Step 4: Mipmap — Multi-Resolution Pixel Access

For RL environments with varying zoom levels, mipmaps pre-compute the image at multiple resolutions:

**Level $k$:** image downsampled by factor $2^k$, with size $(W/2^k) \times (H/2^k)$

**Total storage:** $\sum_{k=0}^{\log_2 \min(W,H)} \frac{WH}{4^k} = WH \cdot \frac{1}{1 - 1/4} = \frac{4}{3}WH$ — only 33% more memory than the original.

**Trilinear interpolation:** Bilinear interpolation within each mipmap level + linear interpolation between levels → smooth transitions across scales.

### Step 5: RL Agent Observation Preprocessing

When building observations for an RL agent:
1. **Downsampling** (e.g., Atari 210×160 → 84×84): Use area averaging (equivalent to box filter + subsample) to avoid aliasing
2. **Cropping:** No interpolation needed, just array slicing
3. **Rotation/affine:** Use bilinear interpolation for speed
4. The preprocessing pipeline itself can be learned by the agent as part of its policy

## Bit Depth Analysis — Dynamic Range and Quantization Theory

The number of bits per pixel fundamentally determines what an image can represent. This section derives the mathematical relationships between bit depth, dynamic range, and perceptual quality.

### Step 1: Dynamic Range Definition

$$\text{DR} = \frac{I_{\max}}{I_{\min}} = 2^k - 1 \approx 2^k$$

for a $k$-bit image where $I_{\min} = 1$ (minimum nonzero value).

**In decibels:** $\text{DR}_{\text{dB}} = 20 \log_{10}(2^k) = 20k \log_{10}(2) \approx 6.02k$ dB

| Bit Depth | Levels | Dynamic Range | DR (dB) |
|:----------|:-------|:-------------|:--------|
| 1 bit | 2 | 1:1 | 0 dB |
| 4 bits | 16 | 15:1 | 24 dB |
| 8 bits | 256 | 255:1 | 48 dB |
| 12 bits | 4096 | 4095:1 | 72 dB |
| 16 bits | 65536 | 65535:1 | 96 dB |

Human vision has approximately 10,000:1 simultaneous dynamic range (~80 dB), so 14-bit capture is needed for full fidelity.

### Step 2: Quantization Noise Analysis

For uniform quantization with step size $\Delta = (I_{\max} - I_{\min}) / (2^k - 1)$:

**Quantization error** $e = x - Q(x)$ is uniformly distributed on $[-\Delta/2, \Delta/2]$:

$$\text{Var}[e] = E[e^2] = \int_{-\Delta/2}^{\Delta/2} e^2 \cdot \frac{1}{\Delta} \, de = \frac{1}{\Delta} \cdot \frac{e^3}{3}\bigg|_{-\Delta/2}^{\Delta/2} = \frac{\Delta^2}{12} \quad \blacksquare$$

### Step 3: Signal-to-Quantization-Noise Ratio (SQNR)

$$\text{SQNR} = 10\log_{10}\frac{\sigma_x^2}{\sigma_e^2}$$

For a full-range signal with $\sigma_x^2 = (2^k)^2 / 12$:
$$\text{SQNR} = 10\log_{10}\frac{(2^k)^2/12}{\Delta^2/12} = 20\log_{10}(2^k) = 20k\log_{10}(2) \approx 6.02k \text{ dB}$$

**The 6 dB per bit rule:** Each additional bit of depth adds approximately 6 dB of SNR, or equivalently doubles the signal-to-noise ratio.

### Step 4: Perceptual Bit Depth Requirements

The human JND (just-noticeable difference) for brightness is approximately 1% (Weber fraction). For a display with maximum luminance $L_{\max}$:

$$\text{Required levels} = \int_0^{L_{\max}} \frac{dL}{0.01 \cdot L} = 100 \cdot \ln\frac{L_{\max}}{L_{\min}} \approx 100 \ln(1000) \approx 690$$

This requires $\lceil \log_2(690) \rceil = 10$ bits in a **logarithmic** encoding, or $\approx 12-14$ bits in linear encoding (due to the wasted levels in dark regions).

### Step 5: RL State Space Implications

The bit depth exponentially affects the RL state space:
$$|S| = (2^k)^{H \times W \times C}$$

**Reducing from 8-bit to 4-bit** per channel reduces state space from $256^{3HW}$ to $16^{3HW}$ — a compression factor of $16^{3HW}$ in state space, at the cost of quantization artifacts. An RL agent must balance representation fidelity against computational tractability.

## 3. Channel Arithmetic — How RL Agents Manipulate Images

### Basic Operations:

**Addition (Brighten):** $I_{out}(x,y) = \min(I_{in}(x,y) + c, \; 255)$

**Multiplication (Contrast):** $I_{out}(x,y) = \text{clip}(\alpha \cdot I_{in}(x,y), \; 0, \; 255)$

**Blending:** $I_{out} = \alpha \cdot I_1 + (1-\alpha) \cdot I_2$

These are the **primitive actions** our RL agent will learn to compose!

In [ ]:
# Demonstrate channel arithmetic as RL actions
base_img = create_sample_image_v2()

def create_sample_image_v2(size=64):
    """Create a more interesting sample image."""
    img = np.zeros((size, size, 3), dtype=np.float32)
    x = np.linspace(-3, 3, size)
    y = np.linspace(-3, 3, size)
    X, Y = np.meshgrid(x, y)
    
    # Create a colorful pattern
    img[:,:,0] = 0.5 + 0.5 * np.sin(X * 2)  # Red: vertical waves
    img[:,:,1] = 0.5 + 0.5 * np.sin(Y * 2)  # Green: horizontal waves
    img[:,:,2] = 0.5 + 0.5 * np.sin(X + Y)  # Blue: diagonal waves
    
    return np.clip(img, 0, 1)

base = create_sample_image_v2()

# RL Actions on the image
actions = {
    'Original': base,
    'Brighten (+0.3)': np.clip(base + 0.3, 0, 1),
    'Darken (-0.3)': np.clip(base - 0.3, 0, 1),
    'High Contrast (×2)': np.clip(base * 2, 0, 1),
    'Low Contrast (×0.5)': np.clip(base * 0.5, 0, 1),
    'Invert (1-I)': 1.0 - base,
    'Red Only': base * np.array([1, 0, 0]),
    'Channel Swap': base[:,:,[2,0,1]],
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, (name, img) in enumerate(actions.items()):
    ax = axes[idx // 4, idx % 4]
    ax.imshow(img)
    ax.set_title(name, fontsize=10)
    ax.axis('off')

plt.suptitle('Channel Arithmetic = RL Actions on Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('channel_arithmetic.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Key Takeaways

| Concept | Mathematical Form | RL Interpretation |
|:--------|:-----------------|:------------------|
| Pixel | $p \in [0, 255]$ | Smallest unit of state |
| Channel | $C \in \{R, G, B\}$ | Feature dimension |
| Neighborhood | $N_k(p)$ | Local observation |
| Arithmetic | $f(I)$ | Primitive action |

---
**Next:** Module 1.3 — Color Spaces (HSV, LAB, YCbCr)

## Channel Correlation and Decorrelation via the KLT

Color channels in natural images are highly correlated. The Karhunen-Loève Transform (KLT) decorrelates them, which is essential for efficient compression and state representation.

### Step 1: Channel Covariance Matrix

For an RGB image, treat each pixel $\mathbf{p} = (R, G, B)^T$ as a 3D random vector. The covariance matrix:

$$\mathbf{C} = E[(\mathbf{p} - \boldsymbol{\mu})(\mathbf{p} - \boldsymbol{\mu})^T] = \begin{bmatrix} \sigma_R^2 & \sigma_{RG} & \sigma_{RB} \\ \sigma_{RG} & \sigma_G^2 & \sigma_{GB} \\ \sigma_{RB} & \sigma_{GB} & \sigma_B^2 \end{bmatrix}$$

**Typical values for natural images:**
$$\mathbf{C}_{\text{typical}} \approx \begin{bmatrix} 4800 & 4200 & 3500 \\ 4200 & 4500 & 3800 \\ 3500 & 3800 & 4100 \end{bmatrix}$$

The high off-diagonal values (correlations $\rho_{RG} \approx 0.90$) indicate massive redundancy.

### Step 2: KLT Derivation

**Goal:** Find a linear transform $\mathbf{y} = \mathbf{W}(\mathbf{p} - \boldsymbol{\mu})$ such that the components of $\mathbf{y}$ are uncorrelated.

**Uncorrelated means:** $E[\mathbf{y}\mathbf{y}^T] = \boldsymbol{\Lambda}$ (diagonal matrix).

$$E[\mathbf{y}\mathbf{y}^T] = \mathbf{W} E[(\mathbf{p}-\boldsymbol{\mu})(\mathbf{p}-\boldsymbol{\mu})^T] \mathbf{W}^T = \mathbf{W}\mathbf{C}\mathbf{W}^T$$

For this to be diagonal: $\mathbf{W}\mathbf{C}\mathbf{W}^T = \boldsymbol{\Lambda}$, which means $\mathbf{W}$ must be the eigenvector matrix of $\mathbf{C}$.

$$\mathbf{C} = \mathbf{W}^T\boldsymbol{\Lambda}\mathbf{W} \quad \text{(eigendecomposition)} \quad \blacksquare$$

### Step 3: Optimal Energy Compaction

**Theorem (KLT optimality):** Among all orthogonal transforms, the KLT maximizes the energy compaction into the fewest components.

**Proof:** The variance of the $i$-th component is $\lambda_i$ (the $i$-th eigenvalue). The total variance is preserved: $\sum_i \lambda_i = \text{tr}(\mathbf{C})$. The eigenvalues are sorted $\lambda_1 \geq \lambda_2 \geq \lambda_3$, so the first component captures the maximum possible variance. By the Courant-Fischer minimax theorem, no other unit vector captures more variance. $\blacksquare$

### Step 4: Practical Channel Decorrelation

For typical natural images, the KLT yields:
- **Component 1** ($\approx 97\%$ of variance): luminance (weighted sum of R, G, B)
- **Component 2** ($\approx 2\%$): red-green chrominance
- **Component 3** ($\approx 1\%$): blue-yellow chrominance

This mirrors the structure of YCbCr — the KLT reveals that luminance/chrominance separation is not arbitrary but is the mathematically optimal decorrelation.

### Step 5: Alpha Compositing — Porter-Duff Operators

When combining images with transparency, the alpha channel controls blending:

$$C_{\text{out}} = \frac{\alpha_A C_A + (1 - \alpha_A)\alpha_B C_B}{\alpha_{\text{out}}}$$

$$\alpha_{\text{out}} = \alpha_A + \alpha_B(1 - \alpha_A) \quad \text{("over" operator)}$$

**Proof of associativity:** $(A \text{ over } B) \text{ over } C = A \text{ over } (B \text{ over } C)$ — this allows compositing layers in any order, which is essential for efficient rendering in RL environments.

The 12 Porter-Duff operators (over, in, out, atop, xor, ...) form an algebra for layer composition, each defined by how source and destination alpha values interact.